In [ ]:
# SPDX-FileCopyrightText: 2024-2026 Dan J. Bower <dbower@eaps.ethz.ch>
# SPDX-FileCopyrightText: 2024-2026 Paolo A. Sossi
#
# SPDX-License-Identifier: Apache-2.0

import logging
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
from jaxtyping import ArrayLike
from molmass import Formula
from scipy.constants import mega

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    FixedActivityConstraint,
    PurePhase,
    ReservoirSpecies,
    ThermodynamicState,
    debug_logger,
    unit_conversion,
)
from atmodeller.sci_utils import SIO2_MOLAR_MASS
from atmodeller.solubility import get_solubility_models

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

# Geological Outgassing

This notebook is available at `notebooks/examples/geological_outgassing.ipynb` and is easiest to obtain by downloading the source code.

*Atmodeller* can perform "geological outgassing", for example, as appears in Gaillard and Scaillet (2014). This notebook reproduces the basic result of Gaillard and Scaillet (2014), Figure 3, using a fixed fO2 constraint.

## Initial setup

In [ ]:
# Gas species
CO_g = ChemicalSpecies.create_gas("CO")
O2_g = ChemicalSpecies.create_gas("O2")
CO2_g = ChemicalSpecies.create_gas("CO2")
H2_g = ChemicalSpecies.create_gas("H2")
H2O_g = ChemicalSpecies.create_gas("H2O")
CH4_g = ChemicalSpecies.create_gas("CH4")
S2_g = ChemicalSpecies.create_gas("S2")
SO2_g = ChemicalSpecies.create_gas("SO2")
H2S_g = ChemicalSpecies.create_gas("H2S")
gas_species = (CO_g, O2_g, CO2_g, H2_g, H2O_g, CH4_g, S2_g, SO2_g, H2S_g)

# Melt species
solubility_models = get_solubility_models()

# Exclude mass of species in melt from mass balance constraints, as they are negligible compared to
# the mass of the silicate melt itself.
include_in_phase_mass = False

# Gaillard and Scaillet (2014) neglect CO solubility (low in basaltic melt), but it is included
# here
CO_d = ReservoirSpecies.create_dissolved(
    "CO",
    solubility=solubility_models["CO_basalt_yoshioka19"],
    include_in_phase_mass=include_in_phase_mass,
)
CO2_d = ReservoirSpecies.create_dissolved(
    "CO2",
    solubility=solubility_models["CO2_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)
H2_d = ReservoirSpecies.create_dissolved(
    "H2",
    solubility=solubility_models["H2_basalt_hirschmann12"],
    include_in_phase_mass=include_in_phase_mass,
)
H2O_d = ReservoirSpecies.create_dissolved(
    "H2O",
    solubility=solubility_models["H2O_basalt_mitchell17"],
    include_in_phase_mass=include_in_phase_mass,
)
# Gaillard and Scaillet (2014) neglect CH4 solubility (low in basaltic melt), but it is included
# here
CH4_d = ReservoirSpecies.create_dissolved(
    "CH4",
    solubility=solubility_models["CH4_basalt_ardia13"],
    include_in_phase_mass=include_in_phase_mass,
)
S2_d = ReservoirSpecies.create_dissolved(
    "S2",
    solubility=solubility_models["S2_basalt_boulliung23"],
    include_in_phase_mass=include_in_phase_mass,
)
melt_species = (CO_d, CO2_d, H2_d, H2O_d, CH4_d, S2_d)

# Condensates
# Allow for graphite if stable, although the result demonstrates that graphite is not stable.
graphite = PurePhase.from_end_member("graphite")
condensate_phases = (graphite,)

## Constraints

Parameters from Gaillard and Scaillet (2014).

In [ ]:
temperature_celsius = 1300
# Note that the Figure 3 inset box gives -1.4 but the caption gives -1.5
dlog10fO2_QFM = -1.5

temperature_K = temperature_celsius + 273.15

# The shift is quoted relative to FMQ, so use the online buffer calculator to convert to get the
# absolute FMQ in log10 units at 1300 C (https://fo2.rses.anu.edu.au/fo2app/)
# log10fO2 below is at temperature = 1302.04081632653 C at 1 bar
log10fO2 = -7.27787591128208

# Account for the shift to get the absolute fO2 in log10 units
log10fO2 += dlog10fO2_QFM

# Whether to export the data to Excel and pickle files, and where
WRITE_OUTPUT = False
OUTPUT_DIRECTORY = Path("output") / "geological_outgassing"

# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

if WRITE_OUTPUT:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

Take the nominal concentrations of C, H, and S from Gaillard and Scaillet (2014) and convert to masses per kilogram of basalt. This is consistent with the default mass of the melt in `ThermodynamicState`, which is 1 kg.

In [ ]:
# Initial volatile abundances (mole fraction) as estimated in undegassed Mid Ocean Ridge Basalts
CO2_initial = 600.0 / mega
H2O_initial = 1000.0 / mega
S_initial = 1000.0 / mega

# Approximate molar mass of basalt as pure SiO2, assuming some heavier and lighter components
# roughly cancel. A more accurate calculation could be performed if desired.
molar_mass_basalt = SIO2_MOLAR_MASS

# Mass fraction of volatiles per unit kg of basalt
C_mass = CO2_initial * Formula("C").mass * unit_conversion.g_to_kg / molar_mass_basalt
H_mass = H2O_initial * Formula("H2").mass * unit_conversion.g_to_kg / molar_mass_basalt
S_mass = S_initial * Formula("S").mass * unit_conversion.g_to_kg / molar_mass_basalt

logger.info("Mass of carbon = %0.5e", C_mass)
logger.info("Mass of hydrogen = %0.5e", H_mass)
logger.info("Mass of sulfur = %0.5e", S_mass)

# Impose the fO2 and volatile masses as fixed constraints
fugacity_constraints = {"O2_g": FixedActivityConstraint(10**log10fO2)}
mass_constraints: dict[str, ArrayLike] = {"C": C_mass, "H": H_mass, "S": S_mass}

# Pressure range in bar, equivalent to Figure 3.
pressure_bar = np.logspace(-6, 3, num=100)

## Model

In [ ]:
state = ThermodynamicState.from_species(
    gas_species,
    temperature=temperature_K,
    pressure=pressure_bar,
    silicate_melt_species=melt_species,
    condensates=condensate_phases,
)

model = EquilibriumModel.from_state(
    state, activity_constraints=fugacity_constraints, mass_constraints=mass_constraints
)

# Solve with an automatic initial guess and default solver settings
output = model.solve_with_default()

# Optionally print solver stats to the logger
output.solver_stats_to_logger()

# Quick look at the solution
# output.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output.to_excel(OUTPUT_DIRECTORY / "geological_outgassing", output_format=output_format)

    # Write the data to a pickle file with dataframes
    output.to_pickle(OUTPUT_DIRECTORY / "geological_outgassing", output_format=output_format)

## Plot

Plot the gas speciation to compare with Gaillard and Scaillet (2014), Figure 3.

In [ ]:
# Get data as dataframes
data = output.to_dataframes("elements_species")

# Plot total pressure on the x-axis
total_pressure = data["state"]["pressure"]

# Pick similar colours to Gaillard and Scaillet (2014) for easier visual comparison
colors = {
    "CO_g": "green",
    "CO2_g": "brown",
    "H2O_g": "darkblue",
    "H2_g": "blue",
    "O2S_g": "red",
    "H2S_g": "pink",
    "S2_g": "gold",
}

fig, ax = plt.subplots()

for species, color in colors.items():
    species_data = data[species]["gas.mole_fraction"]
    ax.loglog(total_pressure, species_data, label=species, color=color)

ax.legend()
ax.set_title("Compare with Gaillard and Scaillet (2014), Figure 3")
ax.set_xlabel("Pressure (bar)")
ax.set_ylabel("Mole fraction in the C-H-O-S gas")
ax.set_ylim(0.001, 1)